# API Externa de Odoo (Entorno de pruebas)
### Conexión con la API

In [25]:
import json
import random
import urllib.request

HOST = '52.47.154.185'
PORT = 8069
DB = 'admin'
USER = 'admin'
PASS = 'clave$1'

def json_rpc(url, method, params):
    data = {
        "jsonrpc": "2.0",
        "method": method,
        "params": params,
        "id": random.randint(0, 1000000000),
    }
    print(json.dumps(data).encode())
    req = urllib.request.Request(url=url, data=json.dumps(data).encode(), headers={
        "Content-Type":"application/json",
    })
    reply = json.loads(urllib.request.urlopen(req).read().decode('UTF-8'))
    if reply.get("error"):
        raise Exception(reply["error"])
    return reply["result"]

def call(url, service, method, *args):
    return json_rpc(url, "call", {"service": service, "method": method, "args": args})

### Login y creacion de usuario "Aitor" para Api App

In [ ]:
url = "http://%s:%s/jsonrpc" % (HOST, PORT)
uid = call(url, "common", "login", DB, USER, PASS)


id_grupo = call(url, "object", "execute", DB, uid, PASS, 'res.groups', 'search', [["name","=","Settings"]])

user_id = call(url, "object", "execute", DB, uid, PASS, 'res.users', 'search', [["login","=","aitor"]])
if not user_id:
    user_id = call(url, "object", "execute", DB, uid, PASS, 'res.users', 'create', {'name': "aitor",
                                                                            'login': "aitor",
                                                                            'email': "aitor",
                                                                            'password': "clave$1",
                                                                            'groups_id': [(6, 0, id_grupo)]})


### Crear jugadores desde csv

In [ ]:
#Método para asignar el valor del selector de lateralidad
def latera(late):
    lateral = "derecha"
    if late != "":
        if late[-5] == "Z":
            lateral = "izquierda"
    return lateral

#Leer el fichero csv de la base de datos original
fichero = open('pruebaBD.csv')
linea = fichero.readline()
campos = []
while linea != "":
    #Se separan los campos por comas y se asignan al atributo del modelo
    campos = linea.split(',')
    call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.jugador', 'create', {"nombre":campos[3],
                                                                                    "apellidos":campos[4],
                                                                                    "equipo":campos[6],
                                                                                    "objetivos":campos[8],
                                                                                    "genero":campos[9],
                                                                                    "posicion":campos[10],
                                                                                    "lateralidad":latera(campos[11])})
    linea = fichero.readline()

b'{"jsonrpc": "2.0", "method": "call", "params": {"service": "object", "method": "execute", "args": ["admin", 2, "clave$1", "stmg_jamboree.jugador", "create", {"nombre": "Thiago ", "apellidos": "Acu\\u00f1a Quiroga ", "equipo": "San martin de valdeiglesias", "objetivos": "\\ud83d\\udcaa\\ud83d\\udcc5  Rendimiento (<6jug). BONO MENSUAL: 4s. en grupo reducido \\u27a1\\ufe0f 80\\u20ac/4 sesiones", "genero": "Masculino", "posicion": "\\ud83c\\udfaf Delantero", "lateralidad": "izquierda"}]}, "id": 474871428}'
b'{"jsonrpc": "2.0", "method": "call", "params": {"service": "object", "method": "execute", "args": ["admin", 2, "clave$1", "stmg_jamboree.jugador", "create", {"nombre": "Manuel ", "apellidos": "Aguado Comendador ", "equipo": "Villaviciosa", "objetivos": "", "genero": "Masculino", "posicion": "\\ud83c\\udfaf Delantero", "lateralidad": "derecha"}]}, "id": 572659263}'
b'{"jsonrpc": "2.0", "method": "call", "params": {"service": "object", "method": "execute", "args": ["admin", 2, "clave$1

### Crear tutores desde csv

In [ ]:
#Leer el fichero csv de la base de datos original
fichero = open('pruebaBDTutor.csv')
linea = fichero.readline()
campos = []
while linea != "":
    #Se separan los campos por comas y se asignan al atributo del modelo
    campos = linea.split(',')
    #Primero se almacena el id del jugador buscando a través del ID interno de Jamboree en la tabla de jugador
    id_jugador = call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.jugador', 'search', [["name","=",campos[0]]])
    
    #Si email se inicia el proceso de creación
    if len(campos[4]) > 1:
        #Se busca si existe el tutor ya en el sistema
        id_tutor = call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.tutor', 'search', [["email","=",campos[4]]])
        print(id_tutor)
        #Si no se ha encontrado ningún id de tutor, se crea uno nuevo
        if id_tutor == []:
            nombre_completo = []
            nombre_completo = campos[2].split(" ")
            nombre = ""
            apellido = ""
            #Manipulación para crear el nombre y los apellidos
            for i in range(len(nombre_completo)):
                if i == 0:
                    nombre = nombre_completo[i]
                else:
                    apellido = apellido + " " +nombre_completo[i]
            #Método de creación
            call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.tutor', 'create', {"nombre":nombre,
                                                                                            "apellidos":apellido,
                                                                                            "email":campos[4],
                                                                                            "telefono":campos[3],
                                                                                            "dni":campos[1],
                                                                                            "jugador_ids":id_jugador})
        #Si id_tutor almacena un id, eso quiere decir que ya existe y que lo que necesita es una modificación para relacionarlo con un nuevo jugador
        else:
            consulta = call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.tutor', 'read', id_tutor,['jugador_ids'])
            if consulta != []:
                jugadores = consulta[0]['jugador_ids']
                id_jug = id_jugador[0]
                if id_jug not in jugadores:
                    jugadores.append(id_jug)
                    call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.tutor', 'write', id_tutor, {"jugador_ids": jugadores})

    #Este codigo es una replica del anterior, desplazando el indice del vector para adaptarlo al segundo tutor del registro        
    if len(campos[9]) > 1:
        id_tutor = call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.tutor', 'search', [["email","=",campos[9]]])
        if id_tutor == []:
            nombre_completo = []
            nombre_completo = campos[7].split(" ")
            nombre = ""
            apellido = ""
            for i in range(len(nombre_completo)):
                if i == 0:
                    nombre = nombre_completo[i]
                elif i == 1:
                    apellido = nombre_completo[i]
                else:
                    apellido = apellido + " " + nombre_completo[i]
            call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.tutor', 'create', {"nombre":nombre,
                                                                                    "apellidos":apellido,
                                                                                    "email":campos[9],
                                                                                    "telefono":campos[8],
                                                                                    "dni":campos[6],
                                                                                    "jugador_ids":id_jugador})
        else:
            consulta = call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.tutor', 'read', id_tutor,['jugador_ids'])
            if consulta != []:
                jugadores = consulta[0]['jugador_ids']
                id_jug = id_jugador[0]
                if id_jug not in jugadores:
                    jugadores.append(id_jug)
                    call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.tutor', 'write', id_tutor, {"jugador_ids": jugadores})
    
    linea = fichero.readline()

b'{"jsonrpc": "2.0", "method": "call", "params": {"service": "object", "method": "execute", "args": ["admin", 2, "clave$1", "stmg_jamboree.jugador", "search", [["name", "=", "JUG_1"]]]}, "id": 296158517}'
b'{"jsonrpc": "2.0", "method": "call", "params": {"service": "object", "method": "execute", "args": ["admin", 2, "clave$1", "stmg_jamboree.tutor", "search", [["email", "=", "tutor1@gmail.com"]]]}, "id": 835541859}'
[]
b'{"jsonrpc": "2.0", "method": "call", "params": {"service": "object", "method": "execute", "args": ["admin", 2, "clave$1", "stmg_jamboree.tutor", "create", {"nombre": "Monica", "apellidos": " quroga casafranca", "email": "tutor1@gmail.com", "telefono": "677378880", "dni": "Y84521654N", "jugador_ids": [1]}]}, "id": 645382174}'
b'{"jsonrpc": "2.0", "method": "call", "params": {"service": "object", "method": "execute", "args": ["admin", 2, "clave$1", "stmg_jamboree.tutor", "search", [["email", "=", "tutor2@gmail.com"]]]}, "id": 44248682}'
b'{"jsonrpc": "2.0", "method": "ca

### Datos demo para App

Para probar la app móvil necesitamos datos de demo de entrenamientos, al estar vinculados a la migración de los datos, en vez de crearlos a través de demo.xml, los creamos al final de esta ejecución.
Además se modifica uno de los tutores para forzar muchos registros y poder probar el scroll en la tabla que genera el calendario de la app móvil.

In [29]:
id_ent = call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'search', [["name","=","SEDE_1_ENT_6"]])
call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'write', id_ent, {"jugador_ids": [1,3,5,7,9]})

id_ent = call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'search', [["name","=","SEDE_1_ENT_26"]])
call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'write', id_ent, {"jugador_ids": [1,3,5,7,9]})

id_ent = call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'search', [["name","=","SEDE_2_ENT_3"]])
call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'write', id_ent, {"jugador_ids": [1,2,3,4,5,6,7,8,9]})

id_ent = call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'search', [["name","=","SEDE_2_ENT_23"]])
call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'write', id_ent, {"jugador_ids": [1,2,3,4,5,6,7,8,9]})

id_ent = call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'search', [["name","=","SEDE_1_ENT_7"]])
call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'write', id_ent, {"jugador_ids": [1,2,3,4,5,6,7,8,9]})

id_ent = call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'search', [["name","=","SEDE_1_ENT_27"]])
call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'write', id_ent, {"jugador_ids": [1,2,3,4,5,6,7,8,9]})

id_ent = call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'search', [["name","=","SEDE_2_ENT_8"]])
call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'write', id_ent, {"jugador_ids": [2,4,6,8]})

id_ent = call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'search', [["name","=","SEDE_2_ENT_28"]])
call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.entrenamiento', 'write', id_ent, {"jugador_ids": [2,4,6,8]})

id_tutor = call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.tutor', 'search', [["name","=","TUTOR_2"]])
call(url, "object", "execute", DB, uid, PASS, 'stmg_jamboree.tutor', 'write', id_tutor, {"jugador_ids": [1,2,3,4,5,6,7,8,9]})

b'{"jsonrpc": "2.0", "method": "call", "params": {"service": "object", "method": "execute", "args": ["admin", 2, "clave$1", "stmg_jamboree.entrenamiento", "search", [["name", "=", "SEDE_1_ENT_6"]]]}, "id": 642927857}'
b'{"jsonrpc": "2.0", "method": "call", "params": {"service": "object", "method": "execute", "args": ["admin", 2, "clave$1", "stmg_jamboree.entrenamiento", "write", [6], {"jugador_ids": [1, 3, 5, 7, 9]}]}, "id": 210667664}'
b'{"jsonrpc": "2.0", "method": "call", "params": {"service": "object", "method": "execute", "args": ["admin", 2, "clave$1", "stmg_jamboree.entrenamiento", "search", [["name", "=", "SEDE_1_ENT_26"]]]}, "id": 247556591}'
b'{"jsonrpc": "2.0", "method": "call", "params": {"service": "object", "method": "execute", "args": ["admin", 2, "clave$1", "stmg_jamboree.entrenamiento", "write", [26], {"jugador_ids": [1, 3, 5, 7, 9]}]}, "id": 711334485}'
b'{"jsonrpc": "2.0", "method": "call", "params": {"service": "object", "method": "execute", "args": ["admin", 2, "cl

True